In [0]:
import json
from pathlib import Path
from urllib.request import Request, urlopen

# Store dataset files in your Unity Catalog volume.
sample_dir = Path(
    "/Volumes/workspace/amazon_pyspark_backup/"
    "raw_files/electronics_smoke_sample"
)
sample_dir.mkdir(parents=True, exist_ok=True)

# Pin the dataset version so our source stays consistent.
revision = "2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e"
base_url = (
    "https://huggingface.co/datasets/"
    f"McAuley-Lab/Amazon-Reviews-2023/resolve/{revision}"
)

files = {
    "reviews.jsonl": "raw/review_categories/Electronics.jsonl",
    "products.jsonl": "raw/meta_categories/meta_Electronics.jsonl",
}

# Read at most 2 MiB from each source file.
max_bytes = 2 * 1024 * 1024

for filename, source_path in files.items():
    destination = sample_dir / filename

    if destination.exists():
        print(f"Already exists: {filename}")
        continue

    request = Request(
        f"{base_url}/{source_path}",
        headers={
            "Range": f"bytes=0-{max_bytes - 1}",
            "Accept-Encoding": "identity",
        },
    )

    with urlopen(request, timeout=60) as response:
        if response.status != 206:
            raise RuntimeError("Server did not accept the partial download.")
        content = response.read(max_bytes)

    # Remove the potentially incomplete final JSON record.
    last_newline = content.rfind(b"\n")
    if last_newline < 0:
        raise RuntimeError(f"No complete records found in {filename}.")

    complete_records = content[:last_newline + 1]
    lines = complete_records.splitlines()

    # Validate every record before saving.
    for line in lines:
        json.loads(line)

    with destination.open("xb") as output:
        output.write(complete_records)

    print(f"Saved {filename}: {len(lines):,} records")

In [0]:
display(dbutils.fs.ls(str(sample_dir)))